# Integrated Financial Statement Modeling (CFA Level 1)

A from-scratch capstone notebook that ties together income statement analysis, balance sheet mechanics, cash flow derivation, and forecasting into a **single, coherent three-statement model**.

**Prerequisites:** All prior notebooks in the Financial Statement Analysis series — Introduction to Financial Statements, Income Statement Analysis, Balance Sheet & Working Capital, Cash Flow Analysis, Ratio Analysis & DuPont, and Earnings Quality.

**Outline**
1. Why build financial models
2. Setup
3. Historical financial data
4. Revenue forecasting
5. Cost structure modeling
6. Working capital modeling
7. Capital expenditure and depreciation
8. Income statement projection
9. Balance sheet projection
10. Cash flow statement projection
11. Circular reference resolution
12. Scenario analysis
13. Monte Carlo simulation
14. Model integrity checks
15. References

---
## 1. Why Build Financial Models

Financial modelling is the practice of creating a **mathematical representation of a company's financial performance** — past, present, and future. The three-statement model (income statement, balance sheet, cash flow statement) is the foundation upon which virtually all corporate finance analysis is built.

### 1.1 Core use cases

| Use Case | Description |
|----------|-------------|
| **Forecasting** | Project future revenues, earnings, and cash flows to estimate where the business is heading |
| **Valuation** | DCF models require projected free cash flows; comparable company analysis needs forward multiples |
| **Credit analysis** | Lenders assess interest coverage, leverage ratios, and debt service ability under stress |
| **Scenario planning** | Management and investors test "what if" questions — recession, new product launch, acquisition |
| **Capital allocation** | Boards decide on dividends, buybacks, and reinvestment by modelling their financial impact |
| **M&A analysis** | Acquirers build models of the target to assess synergies and determine bid price |

### 1.2 The three-statement linkage

The power of an integrated model lies in its **internal consistency**. The three statements are not independent; they form a closed system:

$$\text{Net Income} \xrightarrow{\text{flows into}} \text{Retained Earnings (BS)} \xrightarrow{\text{changes drive}} \text{Cash Flow Statement}$$

More precisely:

- The **income statement** produces net income, which feeds retained earnings on the balance sheet.
- **Balance sheet** changes between periods (e.g., increase in receivables, decrease in payables) drive the operating and investing sections of the cash flow statement.
- The **cash flow statement** reconciles net income back to cash, and the ending cash balance feeds back to the balance sheet.
- **Interest expense** on the income statement depends on debt levels from the balance sheet, creating a **circular reference** that must be resolved iteratively.

> **Key Concept:** An integrated three-statement model is internally self-consistent: every dollar earned, spent, borrowed, or invested appears on all three statements in a way that satisfies the accounting equation $A = L + E$ at every point in time.

### 1.3 Model architecture

A well-structured model separates **inputs** (assumptions) from **calculations** from **outputs**:

1. **Historical data** — 3 to 5 years of actual financial statements
2. **Assumptions / drivers** — growth rates, margins, efficiency ratios, capex intensity
3. **Projection engine** — formulas that translate assumptions into projected statements
4. **Integrity checks** — automated tests that the model balances
5. **Scenario layer** — ability to toggle between base, bull, and bear assumptions

> **CFA Exam Tip:** The CFA curriculum emphasises that a good model is *transparent* (assumptions clearly stated), *flexible* (easy to change inputs), and *internally consistent* (balance sheet balances, cash flow reconciles). When answering exam questions about model design, focus on these three qualities.

### 1.4 Common pitfalls

- **Hard-coded numbers** buried in formulas instead of centralised assumption cells
- **Ignoring circular references** — modelling interest expense as a fixed number rather than linking it to average debt
- **Unbalanced balance sheets** — failing to use a "plug" (cash or revolver) to force $A = L + E$
- **Over-precision** — forecasting to the dollar when the inputs have wide uncertainty bands
- **No sanity checks** — blindly trusting outputs without testing whether implied margins, growth rates, and ratios are realistic

> **Common Mistake:** Beginners often project each statement independently and then wonder why the balance sheet does not balance. The statements *must* be built simultaneously with explicit linkages — you cannot project the income statement in isolation.

---
## 2. Setup

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# Tolerances
ATOL = 1e-8
RTOL = 1e-6

# Colour palette
PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print("Setup complete.")

---
## 3. Historical Financial Data

We define three years of historical financial statements for **Apex Manufacturing Corp**, a fictional mid-cap industrial company. The data is synthetic but calibrated to be realistic for a company with approximately \$2 billion in revenue.

### 3.1 Data design principles

When constructing historical data for a model:

- **Internal consistency** — the balance sheet must balance ($A = L + E$) in every historical year
- **Cash flow derivation** — historical cash flows should be derivable from changes in the balance sheet and income statement
- **Realistic ratios** — margins, turns, and leverage should fall within industry norms for manufacturing (gross margin 30-40%, operating margin 10-15%, asset turnover 0.8-1.2x)

> **Key Concept:** Historical financial data serves two purposes in modelling: (1) it provides the **base year** from which projections begin, and (2) it reveals **trends and ratios** that inform assumptions. A model is only as good as its understanding of the past.

### 3.2 Income statement (historical)

| Line Item | Year 1 | Year 2 | Year 3 |
|-----------|--------|--------|--------|
| Revenue | 1,800 | 1,950 | 2,100 |
| COGS | (1,170) | (1,248) | (1,323) |
| Gross Profit | 630 | 702 | 777 |
| SG&A | (270) | (293) | (315) |
| R&D | (90) | (98) | (105) |
| D&A | (72) | (78) | (84) |
| EBIT | 198 | 233 | 273 |
| Interest Expense | (36) | (33) | (30) |
| EBT | 162 | 200 | 243 |
| Tax (25%) | (41) | (50) | (61) |
| Net Income | 122 | 150 | 182 |

*(All figures in millions)*

### 3.3 Balance sheet (historical)

| Line Item | Year 1 | Year 2 | Year 3 |
|-----------|--------|--------|--------|
| Cash | 120 | 145 | 175 |
| Accounts Receivable | 250 | 271 | 292 |
| Inventory | 195 | 208 | 221 |
| **Current Assets** | **565** | **624** | **688** |
| Net PP&E | 900 | 942 | 978 |
| **Total Assets** | **1,465** | **1,566** | **1,666** |
| Accounts Payable | 130 | 139 | 147 |
| Short-term Debt | 50 | 45 | 40 |
| **Current Liabilities** | **180** | **184** | **187** |
| Long-term Debt | 450 | 415 | 380 |
| **Total Liabilities** | **630** | **599** | **567** |
| Common Equity | 835 | 967 | 1,099 |
| **Total L + E** | **1,465** | **1,566** | **1,666** |

> **Common Mistake:** Many students forget to verify that the balance sheet balances in the historical data before building projections. If Total Assets does not equal Total Liabilities plus Equity in the base year, every projection year will be wrong.

### 3.4 Cash flow statement (historical, indirect method)

The cash flow statement is *derived* from changes in the income statement and balance sheet. We will compute it from first principles.

In [ ]:
# ── Historical Income Statement ($ millions) ──
years_hist = np.array([1, 2, 3])
labels_hist = ['Year 1', 'Year 2', 'Year 3']

revenue_hist      = np.array([1800.0, 1950.0, 2100.0])
cogs_hist         = np.array([1170.0, 1248.0, 1323.0])
gross_profit_hist = revenue_hist - cogs_hist
sga_hist          = np.array([270.0,  293.0,  315.0])
rd_hist           = np.array([90.0,   98.0,   105.0])
da_hist           = np.array([72.0,   78.0,   84.0])
ebit_hist         = gross_profit_hist - sga_hist - rd_hist - da_hist
interest_hist     = np.array([36.0,   33.0,   30.0])
ebt_hist          = ebit_hist - interest_hist
tax_rate          = 0.25
tax_hist          = np.round(ebt_hist * tax_rate, 1)
net_income_hist   = ebt_hist - tax_hist

print("=== Historical Income Statement ($M) ===")
for i, yr in enumerate(labels_hist):
    print(f"\n{yr}:")
    print(f"  Revenue:        {revenue_hist[i]:>8.1f}")
    print(f"  COGS:           {-cogs_hist[i]:>8.1f}")
    print(f"  Gross Profit:   {gross_profit_hist[i]:>8.1f}   ({gross_profit_hist[i]/revenue_hist[i]*100:.1f}%)")
    print(f"  SG&A:           {-sga_hist[i]:>8.1f}")
    print(f"  R&D:            {-rd_hist[i]:>8.1f}")
    print(f"  D&A:            {-da_hist[i]:>8.1f}")
    print(f"  EBIT:           {ebit_hist[i]:>8.1f}   ({ebit_hist[i]/revenue_hist[i]*100:.1f}%)")
    print(f"  Interest:       {-interest_hist[i]:>8.1f}")
    print(f"  EBT:            {ebt_hist[i]:>8.1f}")
    print(f"  Tax:            {-tax_hist[i]:>8.1f}")
    print(f"  Net Income:     {net_income_hist[i]:>8.1f}   ({net_income_hist[i]/revenue_hist[i]*100:.1f}%)")

In [ ]:
# ── Historical Balance Sheet ($M) ──
cash_hist       = np.array([120.0, 145.0, 175.0])
ar_hist         = np.array([250.0, 271.0, 292.0])
inventory_hist  = np.array([195.0, 208.0, 221.0])
current_assets_hist = cash_hist + ar_hist + inventory_hist

ppe_hist        = np.array([900.0, 942.0, 978.0])
total_assets_hist = current_assets_hist + ppe_hist

ap_hist         = np.array([130.0, 139.0, 147.0])
st_debt_hist    = np.array([50.0,  45.0,  40.0])
current_liab_hist = ap_hist + st_debt_hist

lt_debt_hist    = np.array([450.0, 415.0, 380.0])
total_liab_hist = current_liab_hist + lt_debt_hist

equity_hist     = np.array([835.0, 967.0, 1099.0])
total_le_hist   = total_liab_hist + equity_hist

print("=== Historical Balance Sheet ($M) ===")
for i, yr in enumerate(labels_hist):
    print(f"\n{yr}:")
    print(f"  Cash:               {cash_hist[i]:>8.1f}")
    print(f"  Accounts Receivable:{ar_hist[i]:>8.1f}")
    print(f"  Inventory:          {inventory_hist[i]:>8.1f}")
    print(f"  Current Assets:     {current_assets_hist[i]:>8.1f}")
    print(f"  Net PP&E:           {ppe_hist[i]:>8.1f}")
    print(f"  Total Assets:       {total_assets_hist[i]:>8.1f}")
    print(f"  ---")
    print(f"  Accounts Payable:   {ap_hist[i]:>8.1f}")
    print(f"  Short-term Debt:    {st_debt_hist[i]:>8.1f}")
    print(f"  Current Liabilities:{current_liab_hist[i]:>8.1f}")
    print(f"  Long-term Debt:     {lt_debt_hist[i]:>8.1f}")
    print(f"  Total Liabilities:  {total_liab_hist[i]:>8.1f}")
    print(f"  Equity:             {equity_hist[i]:>8.1f}")
    print(f"  Total L+E:          {total_le_hist[i]:>8.1f}")
    bal = total_assets_hist[i] - total_le_hist[i]
    check = 'PASS' if abs(bal) < ATOL else 'FAIL'
    print(f"  Balance check (A-L-E): {bal:.1f} [{check}]")

In [ ]:
# ── Derive Historical Cash Flow Statement ($M) ──
# We need Year 0 balance sheet for change computations.
cash_y0, ar_y0, inv_y0 = 100.0, 230.0, 180.0
ap_y0, st_debt_y0, lt_debt_y0 = 120.0, 55.0, 480.0
ppe_y0 = 870.0

# Capex = D&A + change in net PP&E (since Net PP&E_t = Net PP&E_{t-1} + Capex - D&A)
ppe_prev = np.array([ppe_y0, ppe_hist[0], ppe_hist[1]])
capex_hist = ppe_hist - ppe_prev + da_hist

# Dividends = Equity_{t-1} + Net Income - Equity_t
equity_prev_arr = np.array([713.0, 835.0, 967.0])
dividends_hist = equity_prev_arr + net_income_hist - equity_hist

# Cash from operations (indirect)
delta_ar  = np.diff(np.concatenate([[ar_y0], ar_hist]))
delta_inv = np.diff(np.concatenate([[inv_y0], inventory_hist]))
delta_ap  = np.diff(np.concatenate([[ap_y0], ap_hist]))

cfo_hist = net_income_hist + da_hist - delta_ar - delta_inv + delta_ap

# Cash from investing
cfi_hist = -capex_hist

# Cash from financing
delta_st = np.diff(np.concatenate([[st_debt_y0], st_debt_hist]))
delta_lt = np.diff(np.concatenate([[lt_debt_y0], lt_debt_hist]))
cff_hist = delta_st + delta_lt - dividends_hist

# Net change in cash
net_cash_change = cfo_hist + cfi_hist + cff_hist

print("=== Historical Cash Flow Statement ($M) ===")
for i, yr in enumerate(labels_hist):
    print(f"\n{yr}:")
    print(f"  Net Income:              {net_income_hist[i]:>8.1f}")
    print(f"  + D&A:                   {da_hist[i]:>8.1f}")
    print(f"  - Increase in AR:        {-delta_ar[i]:>8.1f}")
    print(f"  - Increase in Inventory: {-delta_inv[i]:>8.1f}")
    print(f"  + Increase in AP:        {delta_ap[i]:>8.1f}")
    print(f"  Cash from Operations:    {cfo_hist[i]:>8.1f}")
    print(f"  Capital Expenditures:    {-capex_hist[i]:>8.1f}")
    print(f"  Cash from Investing:     {cfi_hist[i]:>8.1f}")
    print(f"  Change in ST Debt:       {delta_st[i]:>8.1f}")
    print(f"  Change in LT Debt:       {delta_lt[i]:>8.1f}")
    print(f"  Dividends Paid:          {-dividends_hist[i]:>8.1f}")
    print(f"  Cash from Financing:     {cff_hist[i]:>8.1f}")
    print(f"  Net Change in Cash:      {net_cash_change[i]:>8.1f}")

# Verify cash reconciliation
cash_prev_arr = np.array([cash_y0, cash_hist[0], cash_hist[1]])
print("\n--- Cash Reconciliation ---")
for i in range(3):
    implied = cash_prev_arr[i] + net_cash_change[i]
    check = 'PASS' if abs(implied - cash_hist[i]) < 0.1 else 'FAIL'
    print(f"  {labels_hist[i]}: Beginning {cash_prev_arr[i]:.1f} + Change "
          f"{net_cash_change[i]:.1f} = {implied:.1f} vs Actual {cash_hist[i]:.1f} [{check}]")

---
## 4. Revenue Forecasting

Revenue is the single most important line item in a financial model — virtually every other item is derived from it, either directly (as a percentage of revenue) or indirectly (through balance sheet items that scale with revenue).

### 4.1 Top-down approach

Start with the **total addressable market (TAM)** and work down:

$$\text{Revenue} = \text{Industry Size} \times \text{Market Share} \times \text{Average Selling Price}$$

This approach is useful when you have macro industry data. It forces the analyst to think about whether growth comes from market expansion or share gains.

### 4.2 Bottom-up approach (segment buildup)

Start with the company's **business segments** and build up:

$$\text{Revenue} = \sum_{s=1}^{S} \text{Units}_s \times \text{Price}_s$$

or equivalently, for a multi-segment company:

$$\text{Revenue} = \sum_{s=1}^{S} \text{Revenue}_{s,t-1} \times (1 + g_s)$$

where $g_s$ is the segment-specific growth rate.

### 4.3 Growth rate method

The simplest approach — apply a **compound annual growth rate** to the base year:

$$\text{Revenue}_t = \text{Revenue}_0 \times (1 + g)^t$$

The growth rate $g$ can be estimated from:
- Historical average growth
- Analyst consensus
- Management guidance
- Regression on a macro variable (e.g., GDP)

### 4.4 Regression-based forecasting

If revenue is correlated with a macroeconomic variable $X$ (e.g., industrial production index), we can estimate:

$$\text{Revenue}_t = \alpha + \beta \cdot X_t + \varepsilon_t$$

Then use forecasted values of $X$ to project revenue.

> **Key Concept:** The best models use multiple revenue forecasting methods and compare them for reasonableness. If your growth-rate method says 12% growth but the regression says 4%, that discrepancy is a signal to investigate further.

> **CFA Exam Tip:** The CFA curriculum distinguishes between **top-down** (macro to micro) and **bottom-up** (company-specific) approaches. Top-down is better for understanding industry dynamics; bottom-up provides more granular, actionable projections. A good analyst uses both.

We implement three methods below and compare them.

In [ ]:
# ── Method 1: Growth rate projection ──
hist_growth = np.diff(revenue_hist) / revenue_hist[:-1]
avg_growth = np.mean(hist_growth)

n_proj = 5
years_proj = np.arange(4, 4 + n_proj)  # Years 4-8
labels_proj = [f'Year {y}' for y in years_proj]

rev_growth_method = revenue_hist[-1] * (1 + avg_growth) ** np.arange(1, n_proj + 1)

print(f"Historical growth rates: {hist_growth}")
print(f"Average growth rate: {avg_growth:.4f} ({avg_growth*100:.2f}%)")
print(f"\nMethod 1 -- Growth Rate Projection:")
for i, yr in enumerate(labels_proj):
    print(f"  {yr}: ${rev_growth_method[i]:,.1f}M")

In [ ]:
# ── Method 2: Regression on macro variable (Industrial Production Index) ──
# Synthetic macro data correlated with revenue
ipi_hist = np.array([98.0, 102.5, 107.0])  # Industrial Production Index
ipi_proj = np.array([110.0, 113.0, 115.5, 118.0, 121.0])  # Forecasted IPI

slope, intercept, r_value, p_value, std_err = stats.linregress(ipi_hist, revenue_hist)
rev_regression_method = intercept + slope * ipi_proj

print(f"Regression: Revenue = {intercept:.1f} + {slope:.2f} * IPI")
print(f"R-squared: {r_value**2:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"\nMethod 2 -- Regression Projection:")
for i, yr in enumerate(labels_proj):
    print(f"  {yr}: ${rev_regression_method[i]:,.1f}M  (IPI = {ipi_proj[i]})")

In [ ]:
# ── Method 3: Segment-level buildup ──
# Apex has 3 segments: Industrial (55%), Automotive (30%), Aerospace (15%)
seg_shares = np.array([0.55, 0.30, 0.15])
seg_names = ['Industrial', 'Automotive', 'Aerospace']
seg_growth = np.array([0.06, 0.08, 0.12])  # Different growth by segment

seg_revenue_base = revenue_hist[-1] * seg_shares

rev_segment_method = np.zeros(n_proj)
print("Method 3 -- Segment Buildup Projection:\n")
for t in range(n_proj):
    seg_rev_t = seg_revenue_base * (1 + seg_growth) ** (t + 1)
    rev_segment_method[t] = seg_rev_t.sum()
    print(f"  {labels_proj[t]}:")
    for s in range(3):
        print(f"    {seg_names[s]:>12s}: ${seg_rev_t[s]:>8.1f}M  (g={seg_growth[s]*100:.0f}%)")
    print(f"    {'Total':>12s}: ${rev_segment_method[t]:>8.1f}M")
    print()

In [ ]:
# ── Compare the three methods ──
fig, ax = plt.subplots(figsize=(10, 6))

all_years = np.concatenate([years_hist, years_proj])
ax.plot(years_hist, revenue_hist, 'ko-', markersize=8, linewidth=2, label='Historical')
ax.plot(years_proj, rev_growth_method, 's--', color=PRIMARY, markersize=7, label='Growth Rate')
ax.plot(years_proj, rev_regression_method, 'D--', color=SECONDARY, markersize=7, label='Regression (IPI)')
ax.plot(years_proj, rev_segment_method, '^--', color=TERTIARY, markersize=7, label='Segment Buildup')

ax.axvline(x=3.5, color='gray', linestyle=':', alpha=0.7)
ax.text(3.55, revenue_hist[-1] * 1.15, 'Forecast\nperiod', fontsize=10, color='gray')
ax.set_xlabel('Year')
ax.set_ylabel('Revenue ($M)')
ax.set_title('Revenue Forecast Comparison -- Three Methods')
ax.legend()
ax.set_xticks(all_years)
ax.set_xticklabels(['Y1','Y2','Y3','Y4','Y5','Y6','Y7','Y8'])
plt.tight_layout()
plt.show()

# Use weighted average for the model
weights = np.array([0.3, 0.3, 0.4])  # Slight preference for segment buildup
revenue_proj = (weights[0] * rev_growth_method +
                weights[1] * rev_regression_method +
                weights[2] * rev_segment_method)
print("Blended revenue forecast (30% growth, 30% regression, 40% segment):")
for i, yr in enumerate(labels_proj):
    print(f"  {yr}: ${revenue_proj[i]:,.1f}M")

---
## 5. Cost Structure Modeling

Once revenue is projected, we model each major expense category. The key insight is that costs have both **fixed** and **variable** components, and understanding the mix determines **operating leverage**.

### 5.1 Fixed vs variable costs

$$\text{Total Cost} = \text{Fixed Cost} + \text{Variable Cost per Unit} \times \text{Units}$$

In percentage-of-revenue terms:

$$\frac{\text{Cost}}{\text{Revenue}} = \frac{\text{Fixed Cost}}{\text{Revenue}} + \text{Variable Ratio}$$

As revenue grows, the fixed-cost ratio *declines* (operating leverage), improving margins.

### 5.2 Operating leverage

The **degree of operating leverage (DOL)** measures how sensitive EBIT is to revenue changes:

$$\text{DOL} = \frac{\%\Delta \text{EBIT}}{\%\Delta \text{Revenue}} = \frac{\text{Contribution Margin}}{\text{EBIT}}$$

A company with high fixed costs (high DOL) sees profits swing more dramatically with revenue changes — amplifying gains in good times and losses in bad times.

### 5.3 Modelling approach

For each cost line, we compute the historical **cost-as-a-percentage-of-revenue**, identify trends, and project forward:

| Cost Item | Historical Range | Projection Assumption |
|-----------|-----------------|----------------------|
| COGS | 63.0% - 65.0% | Gradual improvement due to scale |
| SG&A | 14.5% - 15.0% | Slight leverage from fixed component |
| R&D | 5.0% - 5.0% | Held constant as percentage of revenue |
| D&A | 3.8% - 4.0% | Linked to PP&E balance |

> **Key Concept:** COGS is primarily variable (raw materials, direct labour) while SG&A has a significant fixed component (rent, management salaries). This distinction matters enormously for scenario analysis — in a downturn, a company with mostly variable costs can flex expenses down, while one with mostly fixed costs cannot.

> **CFA Exam Tip:** When the exam asks about operating leverage, remember: high DOL means high fixed costs, which means volatile earnings. A cyclical manufacturer with high DOL is much riskier than a service company with mostly variable costs.

In [ ]:
# ── Historical cost ratios ──
cogs_pct_hist = cogs_hist / revenue_hist
sga_pct_hist  = sga_hist / revenue_hist
rd_pct_hist   = rd_hist / revenue_hist
da_pct_hist   = da_hist / revenue_hist

print("Historical Cost Ratios (% of Revenue):")
print(f"{'':>6} {'COGS':>8} {'SGA':>8} {'R&D':>8} {'D&A':>8}")
for i in range(3):
    print(f"  Y{i+1}: {cogs_pct_hist[i]*100:>7.2f}% {sga_pct_hist[i]*100:>7.2f}% "
          f"{rd_pct_hist[i]*100:>7.2f}% {da_pct_hist[i]*100:>7.2f}%")

print(f"\n  Avg: {cogs_pct_hist.mean()*100:>7.2f}% {sga_pct_hist.mean()*100:>7.2f}% "
      f"{rd_pct_hist.mean()*100:>7.2f}% {da_pct_hist.mean()*100:>7.2f}%")

# DOL calculation
pct_rev_change = np.diff(revenue_hist) / revenue_hist[:-1]
pct_ebit_change = np.diff(ebit_hist) / ebit_hist[:-1]
dol = pct_ebit_change / pct_rev_change
print(f"\nDegree of Operating Leverage:")
for i in range(len(dol)):
    print(f"  Y{i+1} to Y{i+2}: DOL = {dol[i]:.2f}x")

In [ ]:
# ── Project costs ──
# COGS: slight improvement (scale economies) — linear decline from 62.5% to 61.0%
cogs_pct_proj = np.linspace(0.625, 0.610, n_proj)

# SG&A: fixed component of ~$100M + variable ~10% of revenue
sga_fixed = 100.0
sga_variable_pct = 0.10
sga_proj = sga_fixed + sga_variable_pct * revenue_proj

# R&D: constant 5% of revenue
rd_pct_proj = 0.05
rd_proj = rd_pct_proj * revenue_proj

cogs_proj = cogs_pct_proj * revenue_proj
gross_profit_proj = revenue_proj - cogs_proj

print("Projected Cost Structure ($M):")
print(f"{'Year':>6} {'Revenue':>10} {'COGS':>10} {'COGS%':>7} {'GP':>10} {'GP%':>7} {'SGA':>10} {'R&D':>10}")
for i in range(n_proj):
    print(f"  Y{years_proj[i]}: {revenue_proj[i]:>9.1f} {cogs_proj[i]:>9.1f} "
          f"{cogs_pct_proj[i]*100:>6.1f}% {gross_profit_proj[i]:>9.1f} "
          f"{gross_profit_proj[i]/revenue_proj[i]*100:>6.1f}% "
          f"{sga_proj[i]:>9.1f} {rd_proj[i]:>9.1f}")

---
## 6. Working Capital Modeling

Working capital — the difference between current assets and current liabilities — is the lifeblood of operations. To project it, we use **efficiency ratios** derived from historical data.

### 6.1 Key efficiency ratios

**Days Sales Outstanding (DSO)** — how quickly the company collects receivables:
$$\text{DSO} = \frac{\text{Accounts Receivable}}{\text{Revenue}} \times 365$$

**Days Inventory Outstanding (DIO)** — how long inventory sits before being sold:
$$\text{DIO} = \frac{\text{Inventory}}{\text{COGS}} \times 365$$

**Days Payable Outstanding (DPO)** — how long the company takes to pay suppliers:
$$\text{DPO} = \frac{\text{Accounts Payable}}{\text{COGS}} \times 365$$

**Cash Conversion Cycle (CCC):**
$$\text{CCC} = \text{DSO} + \text{DIO} - \text{DPO}$$

The CCC measures how many days it takes to convert a dollar spent on inventory into a dollar collected from customers.

### 6.2 Projection method

To project working capital items, we hold efficiency ratios at their historical averages (or trend them) and solve backwards:

$$\text{AR}_t = \frac{\text{DSO}}{365} \times \text{Revenue}_t$$

$$\text{Inventory}_t = \frac{\text{DIO}}{365} \times \text{COGS}_t$$

$$\text{AP}_t = \frac{\text{DPO}}{365} \times \text{COGS}_t$$

> **Key Concept:** Changes in working capital directly affect operating cash flow. An increase in receivables means the company earned revenue it has not collected — cash flow is *lower* than net income. Conversely, an increase in payables means the company consumed inputs it has not paid for — cash flow is *higher* than net income.

> **CFA Exam Tip:** The cash conversion cycle is a favourite CFA exam topic. A shorter CCC is generally better — it means the company converts inventory to cash faster. But be careful: a very low DPO might mean the company is paying suppliers too quickly, or a very low DIO might indicate stock-out risk.

In [ ]:
# ── Compute historical efficiency ratios ──
dso_hist = ar_hist / revenue_hist * 365
dio_hist = inventory_hist / cogs_hist * 365
dpo_hist = ap_hist / cogs_hist * 365
ccc_hist = dso_hist + dio_hist - dpo_hist

print("Historical Efficiency Ratios (days):")
print(f"{'':>6} {'DSO':>8} {'DIO':>8} {'DPO':>8} {'CCC':>8}")
for i in range(3):
    print(f"  Y{i+1}: {dso_hist[i]:>7.1f} {dio_hist[i]:>7.1f} {dpo_hist[i]:>7.1f} {ccc_hist[i]:>7.1f}")
print(f"  Avg: {dso_hist.mean():>7.1f} {dio_hist.mean():>7.1f} {dpo_hist.mean():>7.1f} {ccc_hist.mean():>7.1f}")

# Use average ratios for projection
dso_proj = dso_hist.mean()
dio_proj = dio_hist.mean()
dpo_proj = dpo_hist.mean()

# Project working capital items
ar_proj  = dso_proj / 365 * revenue_proj
inv_proj = dio_proj / 365 * cogs_proj
ap_proj  = dpo_proj / 365 * cogs_proj

print(f"\nProjected Working Capital ($M):")
print(f"{'Year':>6} {'AR':>10} {'Inventory':>10} {'AP':>10} {'NWC':>10}")
for i in range(n_proj):
    nwc = ar_proj[i] + inv_proj[i] - ap_proj[i]
    print(f"  Y{years_proj[i]}: {ar_proj[i]:>9.1f} {inv_proj[i]:>9.1f} {ap_proj[i]:>9.1f} {nwc:>9.1f}")

In [ ]:
# ── Visualise the Cash Conversion Cycle ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: historical trends
x = np.arange(3)
width = 0.25
ax1.bar(x - width, dso_hist, width, label='DSO', color=PRIMARY)
ax1.bar(x, dio_hist, width, label='DIO', color=SECONDARY)
ax1.bar(x + width, dpo_hist, width, label='DPO', color=TERTIARY)
ax1.plot(x, ccc_hist, 'D-', color=ACCENT, markersize=8, linewidth=2, label='CCC')
ax1.set_xticks(x)
ax1.set_xticklabels(labels_hist)
ax1.set_ylabel('Days')
ax1.set_title('Historical Efficiency Ratios')
ax1.legend()

# Right: projected NWC
nwc_proj = ar_proj + inv_proj - ap_proj
nwc_hist_vals = ar_hist + inventory_hist - ap_hist
all_nwc = np.concatenate([nwc_hist_vals, nwc_proj])
all_x = np.concatenate([years_hist, years_proj])
colours = [PRIMARY]*3 + [SECONDARY]*5
ax2.bar(all_x, all_nwc, color=colours, edgecolor='white', linewidth=0.5)
ax2.axvline(x=3.5, color='gray', linestyle=':', alpha=0.7)
ax2.set_xticks(all_x)
ax2.set_xticklabels([f'Y{y}' for y in all_x])
ax2.set_ylabel('Net Working Capital ($M)')
ax2.set_title('Historical and Projected NWC')

plt.tight_layout()
plt.show()

---
## 7. Capital Expenditure and Depreciation

### 7.1 Maintenance vs growth capex

Capital expenditure can be decomposed into two components:

$$\text{Total Capex} = \text{Maintenance Capex} + \text{Growth Capex}$$

- **Maintenance capex** replaces worn-out assets to sustain current capacity. A common proxy is depreciation expense: $\text{Maintenance Capex} \approx \text{D\&A}$.
- **Growth capex** expands capacity. It depends on the company's strategic plans and revenue growth.

### 7.2 Capex-to-depreciation ratio

The ratio of Capex to D&A is a useful diagnostic:

| Ratio | Interpretation |
|-------|---------------|
| Approximately 1.0 | Company is spending just enough to maintain its asset base |
| Greater than 1.5 | Significant growth investment — capacity is expanding |
| Less than 1.0 | Under-investment — the asset base is shrinking |

### 7.3 Depreciation modelling

For a simple model, depreciation is estimated as:

$$\text{D\&A}_t = \text{Depreciation Rate} \times \frac{\text{Net PP\&E}_{t-1} + \text{Net PP\&E}_t}{2}$$

### 7.4 PP&E roll-forward

$$\text{Net PP\&E}_t = \text{Net PP\&E}_{t-1} + \text{Capex}_t - \text{D\&A}_t$$

> **Key Concept:** Capex and depreciation form a feedback loop with the balance sheet. Capex increases PP&E (investing cash flow), while depreciation decreases PP&E (a non-cash charge that reduces net income but is added back in operating cash flow). Understanding this loop is essential for model integrity.

> **Common Mistake:** Do not confuse depreciation (an accounting allocation of past capex) with current capex. A company can have high depreciation (from past investments) but low current capex (under-investing now). The capex-to-D&A ratio catches this.

In [ ]:
# ── Historical capex analysis ──
capex_to_da = capex_hist / da_hist

print("Historical Capex Analysis ($M):")
print(f"{'':>6} {'Capex':>8} {'D&A':>8} {'Capex/DA':>10} {'Net PP&E':>10}")
for i in range(3):
    print(f"  Y{i+1}: {capex_hist[i]:>7.1f} {da_hist[i]:>7.1f} {capex_to_da[i]:>9.2f}x {ppe_hist[i]:>9.1f}")

# ── Project capex and depreciation ──
# Capex = maintenance (1.0x D&A) + growth (linked to revenue growth)
# We target a capex/D&A ratio of ~1.3x (moderate growth phase)
capex_da_target = 1.30
depr_rate = 0.085  # ~8.5% of average net PP&E

# Iterative: D&A depends on PP&E, PP&E depends on capex, capex depends on D&A
ppe_proj = np.zeros(n_proj)
da_proj = np.zeros(n_proj)
capex_proj = np.zeros(n_proj)

ppe_prev = ppe_hist[-1]
for t in range(n_proj):
    # Estimate D&A from previous PP&E
    da_est = depr_rate * ppe_prev
    # Capex = target ratio * D&A
    capex_est = capex_da_target * da_est
    # New PP&E
    ppe_new = ppe_prev + capex_est - da_est
    # Refine D&A using average PP&E
    da_refined = depr_rate * (ppe_prev + ppe_new) / 2
    capex_refined = capex_da_target * da_refined
    ppe_final = ppe_prev + capex_refined - da_refined

    da_proj[t] = da_refined
    capex_proj[t] = capex_refined
    ppe_proj[t] = ppe_final
    ppe_prev = ppe_final

print(f"\nProjected Capex & Depreciation ($M):")
print(f"{'Year':>6} {'Capex':>10} {'D&A':>10} {'Capex/DA':>10} {'Net PP&E':>10}")
for i in range(n_proj):
    print(f"  Y{years_proj[i]}: {capex_proj[i]:>9.1f} {da_proj[i]:>9.1f} "
          f"{capex_proj[i]/da_proj[i]:>9.2f}x {ppe_proj[i]:>9.1f}")

---
## 8. Income Statement Projection

With revenue, costs, and D&A projected, we can build the complete projected income statement. Note that interest expense will initially use a **placeholder** estimate — we will resolve the circular reference in Section 11.

> **Key Concept:** The income statement projection flows top-down: Revenue minus COGS gives gross profit, minus operating expenses gives EBIT, minus interest gives EBT, minus tax gives net income. Each line connects to assumptions we have already established.

In [ ]:
# ── Build projected income statement ──
# Interest expense placeholder: use Year 3 rate on average total debt
# (Will be refined in circular reference section)
total_debt_hist_y3 = st_debt_hist[-1] + lt_debt_hist[-1]
avg_interest_rate = interest_hist[-1] / total_debt_hist_y3

# For initial projection, assume debt declines linearly
st_debt_proj_init = np.maximum(st_debt_hist[-1] - 5 * np.arange(1, n_proj + 1), 10)
lt_debt_proj_init = np.maximum(lt_debt_hist[-1] - 30 * np.arange(1, n_proj + 1), 200)
total_debt_proj_init = st_debt_proj_init + lt_debt_proj_init

interest_proj_init = avg_interest_rate * total_debt_proj_init

ebit_proj = gross_profit_proj - sga_proj - rd_proj - da_proj
ebt_proj_init = ebit_proj - interest_proj_init
tax_proj_init = np.maximum(ebt_proj_init * tax_rate, 0)
net_income_proj_init = ebt_proj_init - tax_proj_init

print("=== Projected Income Statement (Initial, $M) ===")
print(f"{'':>12} " + " ".join(f"{'Y'+str(y):>10}" for y in years_proj))
print("-" * 70)

rows = [
    ('Revenue', revenue_proj),
    ('COGS', -cogs_proj),
    ('Gross Profit', gross_profit_proj),
    ('SGA', -sga_proj),
    ('R&D', -rd_proj),
    ('D&A', -da_proj),
    ('EBIT', ebit_proj),
    ('Interest', -interest_proj_init),
    ('EBT', ebt_proj_init),
    ('Tax', -tax_proj_init),
    ('Net Income', net_income_proj_init),
]

for name, vals in rows:
    line = f"  {name:<12}" + " ".join(f"{v:>10.1f}" for v in vals)
    print(line)

# Margins
print(f"\n{'GP Margin':<14}" + " ".join(f"{gross_profit_proj[i]/revenue_proj[i]*100:>9.1f}%" for i in range(n_proj)))
print(f"{'EBIT Margin':<14}" + " ".join(f"{ebit_proj[i]/revenue_proj[i]*100:>9.1f}%" for i in range(n_proj)))
print(f"{'Net Margin':<14}" + " ".join(f"{net_income_proj_init[i]/revenue_proj[i]*100:>9.1f}%" for i in range(n_proj)))

---
## 9. Balance Sheet Projection

The balance sheet is the most complex of the three statements to project because it must **balance** — total assets must equal total liabilities plus equity in every projected year. This requires a **plug variable** that absorbs any surplus or shortfall.

### 9.1 The balancing mechanism

After projecting all individual line items:

$$\text{Total Assets} = \text{Cash} + \text{AR} + \text{Inventory} + \text{PP\&E}$$
$$\text{Total L+E} = \text{AP} + \text{ST Debt} + \text{LT Debt} + \text{Equity}$$

If Total Assets exceeds Total L+E, the company needs more financing (debt or equity).
If Total Assets is less than Total L+E, the company has excess cash.

### 9.2 Cash as the plug

The most common approach for a profitable, growing company is to use **cash** as the plug:

$$\text{Cash}_t = \text{Total L+E}_t - \text{AR}_t - \text{Inventory}_t - \text{PP\&E}_t$$

If the plug is negative (the company needs more cash than it has), we switch to a **debt revolver** as the plug.

### 9.3 Equity roll-forward

Retained earnings (and therefore equity) builds through:

$$\text{Equity}_t = \text{Equity}_{t-1} + \text{Net Income}_t - \text{Dividends}_t$$

We assume a **payout ratio** (dividends as a percentage of net income) based on historical behaviour.

> **Key Concept:** The balance sheet plug is the mechanism that ensures internal consistency. Without it, the model will have a "gap" between assets and liabilities that grows larger each year. Professional models always include a plug — usually cash (for surplus) or a revolver (for deficit).

> **Common Mistake:** A common error is to project cash independently (e.g., "cash grows 5% per year") rather than letting it be determined by the model. Cash is the *residual* — what is left after all operating, investing, and financing activities. Projecting it independently breaks the model's internal logic.

In [ ]:
# ── Project balance sheet (initial, before circular resolution) ──
# Dividend payout ratio from historical
dividends_pct = dividends_hist / net_income_hist
avg_payout = dividends_pct.mean()
print(f"Historical payout ratios: {dividends_pct}")
print(f"Average payout ratio: {avg_payout:.4f} ({avg_payout*100:.2f}%)")

# Project equity
dividends_proj_init = avg_payout * net_income_proj_init
equity_proj_init = np.zeros(n_proj)
eq_prev = equity_hist[-1]
for t in range(n_proj):
    equity_proj_init[t] = eq_prev + net_income_proj_init[t] - dividends_proj_init[t]
    eq_prev = equity_proj_init[t]

# Non-cash assets
non_cash_assets_proj = ar_proj + inv_proj + ppe_proj

# Liabilities (excluding cash-side plug)
total_liab_proj_init = ap_proj + st_debt_proj_init + lt_debt_proj_init

# Cash as plug: Cash = Equity + Total Liab - Non-cash assets
cash_proj_init = equity_proj_init + total_liab_proj_init - non_cash_assets_proj

# Total assets
total_assets_proj_init = cash_proj_init + ar_proj + inv_proj + ppe_proj
total_le_proj_init = total_liab_proj_init + equity_proj_init

print(f"\n=== Projected Balance Sheet (Initial, $M) ===")
print(f"{'':>18} " + " ".join(f"{'Y'+str(y):>10}" for y in years_proj))
print("-" * 75)

bs_rows = [
    ('Cash (plug)', cash_proj_init),
    ('Accts Receivable', ar_proj),
    ('Inventory', inv_proj),
    ('Net PP&E', ppe_proj),
    ('Total Assets', total_assets_proj_init),
    ('---', None),
    ('Accts Payable', ap_proj),
    ('ST Debt', st_debt_proj_init),
    ('LT Debt', lt_debt_proj_init),
    ('Total Liabilities', total_liab_proj_init),
    ('Equity', equity_proj_init),
    ('Total L+E', total_le_proj_init),
]

for name, vals in bs_rows:
    if vals is None:
        print(f"  {'---'*20}")
        continue
    line = f"  {name:<18}" + " ".join(f"{v:>10.1f}" for v in vals)
    print(line)

# Balance check
print(f"\n  Balance Check (A - L - E):")
for i in range(n_proj):
    diff = total_assets_proj_init[i] - total_le_proj_init[i]
    check = 'PASS' if abs(diff) < ATOL else 'FAIL'
    print(f"    Y{years_proj[i]}: {diff:.6f} [{check}]")

---
## 10. Cash Flow Statement Projection

The projected cash flow statement is **derived** from the projected income statement and balance sheet — it is not independently forecasted. This ensures internal consistency.

### 10.1 Indirect method structure

$$\text{CFO} = \text{Net Income} + \text{D\&A} - \Delta\text{AR} - \Delta\text{Inventory} + \Delta\text{AP}$$

$$\text{CFI} = -\text{Capex}$$

$$\text{CFF} = \Delta\text{Debt} - \text{Dividends}$$

$$\Delta\text{Cash} = \text{CFO} + \text{CFI} + \text{CFF}$$

> **Key Concept:** The cash flow statement is a *check* on the model — if it does not reconcile to the change in cash on the balance sheet, there is an error somewhere. This is one of the most powerful model integrity tests.

In [ ]:
# ── Project cash flow statement (initial) ──
# Changes in working capital
ar_all = np.concatenate([[ar_hist[-1]], ar_proj])
inv_all = np.concatenate([[inventory_hist[-1]], inv_proj])
ap_all = np.concatenate([[ap_hist[-1]], ap_proj])

delta_ar_proj = np.diff(ar_all)
delta_inv_proj = np.diff(inv_all)
delta_ap_proj = np.diff(ap_all)

# CFO
cfo_proj_init = (net_income_proj_init + da_proj
                 - delta_ar_proj - delta_inv_proj + delta_ap_proj)

# CFI
cfi_proj = -capex_proj

# CFF
st_all = np.concatenate([[st_debt_hist[-1]], st_debt_proj_init])
lt_all = np.concatenate([[lt_debt_hist[-1]], lt_debt_proj_init])
delta_st_proj = np.diff(st_all)
delta_lt_proj = np.diff(lt_all)
cff_proj_init = delta_st_proj + delta_lt_proj - dividends_proj_init

# Net change
net_change_proj_init = cfo_proj_init + cfi_proj + cff_proj_init

print("=== Projected Cash Flow Statement (Initial, $M) ===")
print(f"{'':>22} " + " ".join(f"{'Y'+str(y):>10}" for y in years_proj))
print("-" * 78)

cf_rows = [
    ('Net Income', net_income_proj_init),
    ('+ D&A', da_proj),
    ('- Incr. AR', -delta_ar_proj),
    ('- Incr. Inventory', -delta_inv_proj),
    ('+ Incr. AP', delta_ap_proj),
    ('Cash from Ops', cfo_proj_init),
    ('Capital Expend.', -capex_proj),
    ('Cash from Invest', cfi_proj),
    ('Chg ST Debt', delta_st_proj),
    ('Chg LT Debt', delta_lt_proj),
    ('Dividends', -dividends_proj_init),
    ('Cash from Finance', cff_proj_init),
    ('Net Cash Change', net_change_proj_init),
]

for name, vals in cf_rows:
    line = f"  {name:<22}" + " ".join(f"{v:>10.1f}" for v in vals)
    print(line)

# Verify reconciliation
cash_all_init = np.concatenate([[cash_hist[-1]], cash_proj_init])
actual_change = np.diff(cash_all_init)
print(f"\n  Cash Reconciliation:")
for i in range(n_proj):
    print(f"    Y{years_proj[i]}: Computed change = {net_change_proj_init[i]:.1f}, "
          f"Actual change = {actual_change[i]:.1f}, "
          f"Diff = {net_change_proj_init[i] - actual_change[i]:.6f}")

---
## 11. Circular Reference Resolution

### 11.1 The circularity problem

In an integrated model, there is a fundamental **circular dependency**:

1. **Interest expense** depends on the level of **debt** (and cash).
2. **Debt** (or the cash plug) depends on **cash flow**, which determines whether the company has a surplus or deficit.
3. **Cash flow** depends on **net income**, which depends on **interest expense**.

$$\text{Interest} \rightarrow \text{Net Income} \rightarrow \text{Cash Flow} \rightarrow \text{Debt/Cash} \rightarrow \text{Interest}$$

This is a genuine mathematical circularity — each variable depends on the others.

### 11.2 Resolution methods

**Method 1: Iterative convergence** (what we implement)

Start with an initial guess for interest expense, compute the full model, derive the implied debt/cash levels, recompute interest expense, and repeat until convergence:

$$\text{Interest}^{(k+1)} = r \times \frac{\text{Debt}^{(k)}_{t-1} + \text{Debt}^{(k)}_t}{2}$$

This converges because the feedback loop is **contractive** — a one-dollar increase in interest expense reduces net income by $0.75 (after tax), which changes cash by a similar amount, which changes debt/cash, which changes interest by only a fraction of a dollar.

**Method 2: Algebraic solution**

For simple models, you can solve the system of equations simultaneously. However, this becomes unwieldy as the model grows.

**Method 3: Previous-period debt**

Use the *beginning-of-period* debt balance (which is known) to compute interest, breaking the circularity. This is simpler but less accurate.

> **Key Concept:** The circular reference is not a bug — it reflects economic reality. Interest expense genuinely depends on debt levels, which depend on cash flows, which depend on interest expense. A model that ignores this circularity will have a systematic bias.

> **CFA Exam Tip:** The CFA curriculum notes that circular references in financial models should be resolved through iteration. In Excel, this means enabling iterative calculations. In our Python model, we implement this explicitly with a convergence loop.

> **Common Mistake:** Some modellers "break" the circularity by hard-coding interest expense rather than linking it to debt. While this avoids the iteration, it means the model's interest expense is inconsistent with its debt levels — a violation of internal consistency.

In [ ]:
def build_integrated_model(revenue, cogs_pct, sga_fixed, sga_var_pct, rd_pct,
                           depr_rate, capex_da_ratio, tax_rate, avg_int_rate,
                           payout_ratio, dso, dio, dpo,
                           st_debt_schedule, lt_debt_schedule,
                           hist_data, n_years=5, max_iter=100, tol=1e-6):
    '''Build a fully integrated three-statement model with circular reference resolution.

    Parameters: revenue, cogs_pct, sga_fixed, sga_var_pct, rd_pct, depr_rate,
    capex_da_ratio, tax_rate, avg_int_rate, payout_ratio, dso, dio, dpo,
    st_debt_schedule, lt_debt_schedule, hist_data, n_years, max_iter, tol.

    Returns: dict with all projected financial statement arrays.'''
    n = n_years

    # Unpack historical base-year values
    ppe_base = hist_data['ppe']
    equity_base = hist_data['equity']
    ar_base = hist_data['ar']
    inv_base = hist_data['inventory']
    ap_base = hist_data['ap']
    cash_base = hist_data['cash']
    st_debt_base = hist_data['st_debt']
    lt_debt_base = hist_data['lt_debt']

    # ── Items that do not depend on circular reference ──
    cogs = cogs_pct * revenue
    gross_profit = revenue - cogs
    sga = sga_fixed + sga_var_pct * revenue
    rd = rd_pct * revenue

    # Working capital
    ar = dso / 365 * revenue
    inv = dio / 365 * cogs
    ap = dpo / 365 * cogs

    # PP&E and D&A (iterative within themselves)
    ppe = np.zeros(n)
    da = np.zeros(n)
    capex = np.zeros(n)
    ppe_prev_val = ppe_base
    for t in range(n):
        da_est = depr_rate * ppe_prev_val
        capex_est = capex_da_ratio * da_est
        ppe_new = ppe_prev_val + capex_est - da_est
        da_ref = depr_rate * (ppe_prev_val + ppe_new) / 2
        capex_ref = capex_da_ratio * da_ref
        ppe[t] = ppe_prev_val + capex_ref - da_ref
        da[t] = da_ref
        capex[t] = capex_ref
        ppe_prev_val = ppe[t]

    ebit = gross_profit - sga - rd - da

    # ── Circular reference iteration ──
    # Initial guess: interest = 0
    interest = np.zeros(n)

    for iteration in range(max_iter):
        interest_old = interest.copy()

        # Income statement
        ebt = ebit - interest
        tax = np.maximum(ebt * tax_rate, 0)
        net_income = ebt - tax
        dividends = payout_ratio * net_income

        # Equity roll-forward
        equity = np.zeros(n)
        eq_prev = equity_base
        for t in range(n):
            equity[t] = eq_prev + net_income[t] - dividends[t]
            eq_prev = equity[t]

        # Balance sheet: cash as plug
        non_cash = ar + inv + ppe
        total_liab = ap + st_debt_schedule + lt_debt_schedule
        cash = equity + total_liab - non_cash

        # Recompute interest on average debt
        total_debt = st_debt_schedule + lt_debt_schedule
        total_debt_prev = np.concatenate([[st_debt_base + lt_debt_base], total_debt[:-1]])
        avg_debt = (total_debt_prev + total_debt) / 2

        # Interest income on cash (at a lower rate, e.g., 1/3 of debt rate)
        cash_prev = np.concatenate([[cash_base], cash[:-1]])
        avg_cash = (cash_prev + cash) / 2
        interest_on_debt = avg_int_rate * avg_debt
        interest_income = (avg_int_rate / 3) * np.maximum(avg_cash, 0)
        interest = interest_on_debt - interest_income  # Net interest expense

        # Check convergence
        if np.max(np.abs(interest - interest_old)) < tol:
            break

    # Cash flow statement
    delta_ar = np.diff(np.concatenate([[ar_base], ar]))
    delta_inv = np.diff(np.concatenate([[inv_base], inv]))
    delta_ap = np.diff(np.concatenate([[ap_base], ap]))

    cfo = net_income + da - delta_ar - delta_inv + delta_ap
    cfi = -capex
    delta_st = np.diff(np.concatenate([[st_debt_base], st_debt_schedule]))
    delta_lt = np.diff(np.concatenate([[lt_debt_base], lt_debt_schedule]))
    cff = delta_st + delta_lt - dividends
    net_cash_change = cfo + cfi + cff

    total_assets = cash + ar + inv + ppe
    total_le = total_liab + equity

    return {
        'revenue': revenue, 'cogs': cogs, 'gross_profit': gross_profit,
        'sga': sga, 'rd': rd, 'da': da, 'ebit': ebit,
        'interest': interest, 'ebt': ebt, 'tax': tax,
        'net_income': net_income, 'dividends': dividends,
        'cash': cash, 'ar': ar, 'inventory': inv, 'ppe': ppe,
        'total_assets': total_assets,
        'ap': ap, 'st_debt': st_debt_schedule, 'lt_debt': lt_debt_schedule,
        'total_liab': total_liab, 'equity': equity, 'total_le': total_le,
        'capex': capex,
        'cfo': cfo, 'cfi': cfi, 'cff': cff, 'net_cash_change': net_cash_change,
        'delta_ar': delta_ar, 'delta_inv': delta_inv, 'delta_ap': delta_ap,
        'iterations': iteration + 1,
    }

print("Integrated model function defined.")

In [ ]:
# ── Run the integrated model (base case) ──
hist_data = {
    'ppe': ppe_hist[-1], 'equity': equity_hist[-1],
    'ar': ar_hist[-1], 'inventory': inventory_hist[-1],
    'ap': ap_hist[-1], 'cash': cash_hist[-1],
    'st_debt': st_debt_hist[-1], 'lt_debt': lt_debt_hist[-1],
}

base = build_integrated_model(
    revenue=revenue_proj,
    cogs_pct=cogs_pct_proj,
    sga_fixed=sga_fixed, sga_var_pct=sga_variable_pct,
    rd_pct=rd_pct_proj,
    depr_rate=depr_rate, capex_da_ratio=capex_da_target,
    tax_rate=tax_rate, avg_int_rate=avg_interest_rate,
    payout_ratio=avg_payout,
    dso=dso_proj, dio=dio_proj, dpo=dpo_proj,
    st_debt_schedule=st_debt_proj_init,
    lt_debt_schedule=lt_debt_proj_init,
    hist_data=hist_data,
)

print(f"Circular reference resolved in {base['iterations']} iterations.\n")

# Print final income statement
print("=== FINAL Projected Income Statement ($M) ===")
print(f"{'':>14} " + " ".join(f"{'Y'+str(y):>10}" for y in years_proj))
print("-" * 70)
for name, key in [('Revenue','revenue'), ('COGS','cogs'), ('Gross Profit','gross_profit'),
                   ('SGA','sga'), ('R&D','rd'), ('D&A','da'), ('EBIT','ebit'),
                   ('Interest','interest'), ('EBT','ebt'), ('Tax','tax'),
                   ('Net Income','net_income')]:
    sign = -1 if key in ('cogs','sga','rd','da','interest','tax') else 1
    vals = base[key] * sign
    print(f"  {name:<14}" + " ".join(f"{v:>10.1f}" for v in vals))

In [ ]:
# Print final balance sheet
print("=== FINAL Projected Balance Sheet ($M) ===")
print(f"{'':>18} " + " ".join(f"{'Y'+str(y):>10}" for y in years_proj))
print("-" * 75)
for name, key in [('Cash','cash'), ('Accts Receivable','ar'), ('Inventory','inventory'),
                   ('Net PP&E','ppe'), ('Total Assets','total_assets'),
                   ('---', None),
                   ('Accts Payable','ap'), ('ST Debt','st_debt'), ('LT Debt','lt_debt'),
                   ('Total Liabilities','total_liab'), ('Equity','equity'),
                   ('Total L+E','total_le')]:
    if key is None:
        print(f"  {'---'*20}")
        continue
    print(f"  {name:<18}" + " ".join(f"{v:>10.1f}" for v in base[key]))

print(f"\n  Balance Check (A - L - E):")
for i in range(n_proj):
    diff = base['total_assets'][i] - base['total_le'][i]
    check = 'PASS' if abs(diff) < ATOL else 'FAIL'
    print(f"    Y{years_proj[i]}: {diff:.10f} [{check}]")

In [ ]:
# Print final cash flow statement
print("=== FINAL Projected Cash Flow Statement ($M) ===")
print(f"{'':>22} " + " ".join(f"{'Y'+str(y):>10}" for y in years_proj))
print("-" * 78)
for name, key, sign in [
    ('Net Income', 'net_income', 1), ('+ D&A', 'da', 1),
    ('- Incr. AR', 'delta_ar', -1), ('- Incr. Inv', 'delta_inv', -1),
    ('+ Incr. AP', 'delta_ap', 1), ('Cash from Ops', 'cfo', 1),
    ('Capex', 'capex', -1), ('Cash from Invest', 'cfi', 1),
    ('Cash from Finance', 'cff', 1), ('Net Cash Change', 'net_cash_change', 1),
]:
    vals = base[key] * sign
    print(f"  {name:<22}" + " ".join(f"{v:>10.1f}" for v in vals))

# Verify
cash_all = np.concatenate([[cash_hist[-1]], base['cash']])
actual_delta = np.diff(cash_all)
print(f"\n  Reconciliation:")
for i in range(n_proj):
    diff = base['net_cash_change'][i] - actual_delta[i]
    print(f"    Y{years_proj[i]}: CF change = {base['net_cash_change'][i]:.2f}, "
          f"BS change = {actual_delta[i]:.2f}, Diff = {diff:.8f}")

---
## 12. Scenario Analysis

### 12.1 Why scenarios matter

A single-point forecast creates a false sense of precision. In reality, the future is uncertain, and modellers should explore a **range of outcomes**.

The standard approach is to define three scenarios:

| Scenario | Description | Revenue Growth | COGS Adjustment | Capex/D&A |
|----------|-------------|---------------|-----------------|-----------|
| **Bull** | Strong economy, market share gains | +2pp above base | -1pp below base | 1.5x |
| **Base** | Continuation of recent trends | As projected | As projected | 1.3x |
| **Bear** | Recession, margin compression | -3pp below base | +2pp above base | 1.1x |

### 12.2 Fan charts

A **fan chart** displays the range of outcomes as coloured bands, with the base case in the centre and increasingly extreme scenarios in lighter shades. This is a powerful communication tool for boards and investors.

> **Key Concept:** Scenario analysis forces the modeller to think about *what could go wrong* (and what could go right). It transforms a model from a single prediction into a **decision tool** that shows the sensitivity of outcomes to key assumptions.

> **CFA Exam Tip:** The CFA curriculum distinguishes between **scenario analysis** (discrete cases with specific assumptions) and **sensitivity analysis** (varying one input at a time). Both are tested. Scenario analysis changes multiple inputs simultaneously to tell a coherent "story" about a possible future state.

In [ ]:
# ── Define scenario assumptions ──
# Bull: higher growth, lower costs, more investment
bull_growth_adj = 0.02  # +2pp revenue growth
bull_cogs_adj = -0.01   # -1pp COGS ratio
bull_capex_da = 1.50

# Bear: lower growth, higher costs, less investment
bear_growth_adj = -0.03
bear_cogs_adj = 0.02
bear_capex_da = 1.10

# Compute scenario revenues
base_growth_rates = np.diff(np.concatenate([[revenue_hist[-1]], revenue_proj])) / \
                    np.concatenate([[revenue_hist[-1]], revenue_proj[:-1]])

bull_revenue = np.zeros(n_proj)
bear_revenue = np.zeros(n_proj)
rev_prev_bull = revenue_hist[-1]
rev_prev_bear = revenue_hist[-1]
for t in range(n_proj):
    bull_revenue[t] = rev_prev_bull * (1 + base_growth_rates[t] + bull_growth_adj)
    bear_revenue[t] = rev_prev_bear * (1 + max(base_growth_rates[t] + bear_growth_adj, -0.05))
    rev_prev_bull = bull_revenue[t]
    rev_prev_bear = bear_revenue[t]

# Run bull model
bull = build_integrated_model(
    revenue=bull_revenue,
    cogs_pct=cogs_pct_proj + bull_cogs_adj,
    sga_fixed=sga_fixed, sga_var_pct=sga_variable_pct,
    rd_pct=rd_pct_proj, depr_rate=depr_rate, capex_da_ratio=bull_capex_da,
    tax_rate=tax_rate, avg_int_rate=avg_interest_rate,
    payout_ratio=avg_payout, dso=dso_proj, dio=dio_proj, dpo=dpo_proj,
    st_debt_schedule=st_debt_proj_init, lt_debt_schedule=lt_debt_proj_init,
    hist_data=hist_data,
)

# Run bear model
bear = build_integrated_model(
    revenue=bear_revenue,
    cogs_pct=cogs_pct_proj + bear_cogs_adj,
    sga_fixed=sga_fixed, sga_var_pct=sga_variable_pct,
    rd_pct=rd_pct_proj, depr_rate=depr_rate, capex_da_ratio=bear_capex_da,
    tax_rate=tax_rate, avg_int_rate=avg_interest_rate,
    payout_ratio=avg_payout, dso=dso_proj, dio=dio_proj, dpo=dpo_proj,
    st_debt_schedule=st_debt_proj_init, lt_debt_schedule=lt_debt_proj_init,
    hist_data=hist_data,
)

print("Scenario Summary -- Net Income ($M):")
print(f"{'Year':>6} {'Bear':>10} {'Base':>10} {'Bull':>10}")
for i in range(n_proj):
    print(f"  Y{years_proj[i]}: {bear['net_income'][i]:>9.1f} "
          f"{base['net_income'][i]:>9.1f} {bull['net_income'][i]:>9.1f}")

In [ ]:
# ── Fan chart visualisation ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = [
    ('Revenue', 'revenue', revenue_hist),
    ('Net Income', 'net_income', net_income_hist),
    ('EBIT Margin (%)', None, ebit_hist / revenue_hist * 100),
    ('Free Cash Flow', None, cfo_hist + cfi_hist),
]

for ax, (title, key, hist_vals) in zip(axes.flat, metrics):
    if title == 'EBIT Margin (%)':
        base_vals = base['ebit'] / base['revenue'] * 100
        bull_vals = bull['ebit'] / bull['revenue'] * 100
        bear_vals = bear['ebit'] / bear['revenue'] * 100
    elif title == 'Free Cash Flow':
        base_vals = base['cfo'] + base['cfi']
        bull_vals = bull['cfo'] + bull['cfi']
        bear_vals = bear['cfo'] + bear['cfi']
    else:
        base_vals = base[key]
        bull_vals = bull[key]
        bear_vals = bear[key]

    # Historical
    ax.plot(years_hist, hist_vals, 'ko-', markersize=6, linewidth=2, label='Historical')

    # Fan: shade between bear and bull
    ax.fill_between(years_proj, bear_vals, bull_vals, alpha=0.15, color=PRIMARY)
    ax.fill_between(years_proj,
                    base_vals - 0.3*(base_vals - bear_vals),
                    base_vals + 0.3*(bull_vals - base_vals),
                    alpha=0.3, color=PRIMARY)

    ax.plot(years_proj, base_vals, 's-', color=PRIMARY, markersize=5, label='Base')
    ax.plot(years_proj, bull_vals, '^--', color=TERTIARY, markersize=5, alpha=0.7, label='Bull')
    ax.plot(years_proj, bear_vals, 'v--', color=SECONDARY, markersize=5, alpha=0.7, label='Bear')

    ax.axvline(x=3.5, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xticks(np.concatenate([years_hist, years_proj]))
    ax.set_xticklabels([f'Y{y}' for y in np.concatenate([years_hist, years_proj])], fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle('Scenario Analysis -- Fan Charts', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 13. Monte Carlo Simulation

### 13.1 Beyond three scenarios

Scenario analysis considers a handful of discrete cases. **Monte Carlo simulation** goes further by randomly sampling thousands of possible futures, each defined by a different set of input assumptions drawn from probability distributions.

### 13.2 Methodology

For each simulation $i = 1, \ldots, N$:

1. **Draw random assumptions:**
   - Revenue growth rate: $g_i \sim \mathcal{N}(\mu_g, \sigma_g^2)$
   - COGS ratio adjustment: $c_i \sim \mathcal{N}(0, \sigma_c^2)$
   - Capex/D&A ratio: $k_i \sim \mathcal{N}(\mu_k, \sigma_k^2)$

2. **Run the full integrated model** with these assumptions.

3. **Record outputs** of interest (e.g., terminal-year free cash flow, ending equity).

After $N$ simulations, we have a **distribution** of outcomes, from which we can compute percentiles, expected values, and risk measures.

### 13.3 Sensitivity analysis

A **tornado chart** ranks the input variables by their impact on a key output. For each input, we compute the output at the 10th and 90th percentile of that input while holding others at their median, revealing which assumptions matter most.

> **Key Concept:** Monte Carlo simulation converts a deterministic model into a probabilistic one. Instead of asking "what is our forecast?", we ask "what is the *distribution* of possible outcomes?" This is fundamentally more honest and more useful for decision-making.

> **CFA Exam Tip:** Monte Carlo simulation is covered in the CFA curriculum under quantitative methods. The key exam points are: (1) it requires specifying probability distributions for inputs, (2) it produces a distribution of outputs, not a point estimate, and (3) the quality of results depends entirely on the quality of input assumptions.

> **Common Mistake:** Running Monte Carlo with independent random draws when the inputs are actually correlated. If revenue growth and COGS ratio are correlated (they often are — booms raise both revenue and input costs), ignoring this correlation will understate the tails of the distribution.

In [ ]:
# ── Monte Carlo simulation ──
N_SIM = 10_000

# Random parameters
growth_mean = avg_growth
growth_std = 0.03
cogs_adj_std = 0.015
capex_da_mean = 1.30
capex_da_std = 0.15

# Storage
terminal_fcf = np.zeros(N_SIM)
terminal_ni = np.zeros(N_SIM)
terminal_revenue = np.zeros(N_SIM)

# Store parameter draws for sensitivity analysis
param_draws = np.zeros((N_SIM, 3))  # growth, cogs_adj, capex_da

print(f"Running {N_SIM:,} Monte Carlo simulations...")

for sim in range(N_SIM):
    # Draw random parameters
    g_draw = rng.normal(growth_mean, growth_std)
    c_draw = rng.normal(0, cogs_adj_std)
    k_draw = np.clip(rng.normal(capex_da_mean, capex_da_std), 0.8, 2.0)

    param_draws[sim] = [g_draw, c_draw, k_draw]

    # Build revenue with random growth
    rev_sim = revenue_hist[-1] * (1 + g_draw) ** np.arange(1, n_proj + 1)

    # Run model
    try:
        result = build_integrated_model(
            revenue=rev_sim,
            cogs_pct=cogs_pct_proj + c_draw,
            sga_fixed=sga_fixed, sga_var_pct=sga_variable_pct,
            rd_pct=rd_pct_proj, depr_rate=depr_rate, capex_da_ratio=k_draw,
            tax_rate=tax_rate, avg_int_rate=avg_interest_rate,
            payout_ratio=avg_payout, dso=dso_proj, dio=dio_proj, dpo=dpo_proj,
            st_debt_schedule=st_debt_proj_init, lt_debt_schedule=lt_debt_proj_init,
            hist_data=hist_data,
        )
        terminal_fcf[sim] = result['cfo'][-1] + result['cfi'][-1]
        terminal_ni[sim] = result['net_income'][-1]
        terminal_revenue[sim] = result['revenue'][-1]
    except Exception:
        terminal_fcf[sim] = np.nan
        terminal_ni[sim] = np.nan
        terminal_revenue[sim] = np.nan

# Remove any failed simulations
valid = ~np.isnan(terminal_fcf)
terminal_fcf = terminal_fcf[valid]
terminal_ni = terminal_ni[valid]
terminal_revenue = terminal_revenue[valid]
param_draws = param_draws[valid]

print(f"Completed {valid.sum():,} valid simulations.")
print(f"\nTerminal-Year FCF Distribution ($M):")
print(f"  Mean:   ${terminal_fcf.mean():>8.1f}M")
print(f"  Median: ${np.median(terminal_fcf):>8.1f}M")
print(f"  Std:    ${terminal_fcf.std():>8.1f}M")
pcts = [5, 25, 75, 95]
for p in pcts:
    print(f"  {p}th pct: ${np.percentile(terminal_fcf, p):>8.1f}M")

In [ ]:
# ── Histogram of terminal-year FCF ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, data, title, color in zip(
    axes,
    [terminal_revenue, terminal_ni, terminal_fcf],
    ['Terminal Revenue', 'Terminal Net Income', 'Terminal FCF'],
    [PRIMARY, TERTIARY, SECONDARY]
):
    ax.hist(data, bins=80, color=color, alpha=0.7, edgecolor='white', linewidth=0.3)
    mean_val = np.mean(data)
    p5_val = np.percentile(data, 5)
    p95_val = np.percentile(data, 95)
    ax.axvline(mean_val, color='black', linestyle='--', linewidth=1.5,
               label=f'Mean: ${mean_val:.0f}M')
    ax.axvline(p5_val, color='red', linestyle=':', linewidth=1.2,
               label=f'5th: ${p5_val:.0f}M')
    ax.axvline(p95_val, color='red', linestyle=':', linewidth=1.2,
               label=f'95th: ${p95_val:.0f}M')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('$M')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

plt.suptitle('Monte Carlo Results -- ' + f'{valid.sum():,} Simulations (Year {years_proj[-1]})',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Tornado sensitivity chart ──
# For each parameter, compute FCF at its 10th and 90th percentile
# while holding others at median

param_names = ['Revenue Growth', 'COGS Adj.', 'Capex/D&A']
param_medians = np.median(param_draws, axis=0)

tornado_lo = np.zeros(3)
tornado_hi = np.zeros(3)
base_fcf = np.median(terminal_fcf)

for p in range(3):
    p10 = np.percentile(param_draws[:, p], 10)
    p90 = np.percentile(param_draws[:, p], 90)

    for pval, store_idx in [(p10, 0), (p90, 1)]:
        params = param_medians.copy()
        params[p] = pval
        g, c, k = params

        rev_sim = revenue_hist[-1] * (1 + g) ** np.arange(1, n_proj + 1)
        result = build_integrated_model(
            revenue=rev_sim,
            cogs_pct=cogs_pct_proj + c,
            sga_fixed=sga_fixed, sga_var_pct=sga_variable_pct,
            rd_pct=rd_pct_proj, depr_rate=depr_rate, capex_da_ratio=k,
            tax_rate=tax_rate, avg_int_rate=avg_interest_rate,
            payout_ratio=avg_payout, dso=dso_proj, dio=dio_proj, dpo=dpo_proj,
            st_debt_schedule=st_debt_proj_init, lt_debt_schedule=lt_debt_proj_init,
            hist_data=hist_data,
        )
        fcf_val = result['cfo'][-1] + result['cfi'][-1]
        if store_idx == 0:
            tornado_lo[p] = fcf_val
        else:
            tornado_hi[p] = fcf_val

# Sort by range
ranges = tornado_hi - tornado_lo
sort_idx = np.argsort(ranges)

fig, ax = plt.subplots(figsize=(10, 5))
y_pos = np.arange(len(param_names))

for i, idx in enumerate(sort_idx):
    ax.barh(i, tornado_hi[idx] - base_fcf, left=base_fcf, height=0.5,
            color=TERTIARY, alpha=0.8)
    ax.barh(i, tornado_lo[idx] - base_fcf, left=base_fcf, height=0.5,
            color=SECONDARY, alpha=0.8)
    ax.text(tornado_hi[idx] + 2, i, f'${tornado_hi[idx]:.0f}M', va='center', fontsize=10)
    ax.text(tornado_lo[idx] - 2, i, f'${tornado_lo[idx]:.0f}M', va='center',
            ha='right', fontsize=10)

ax.set_yticks(y_pos)
ax.set_yticklabels([param_names[idx] for idx in sort_idx])
ax.axvline(base_fcf, color='black', linestyle='-', linewidth=1.5)
ax.set_xlabel('Terminal-Year FCF ($M)')
ax.set_title('Tornado Chart -- Sensitivity of FCF to Input Assumptions', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nBase case FCF (median): ${base_fcf:.1f}M")
print(f"\nSensitivity ranges (P10 to P90):")
for idx in reversed(sort_idx):
    print(f"  {param_names[idx]:>18s}: ${tornado_lo[idx]:.1f}M to ${tornado_hi[idx]:.1f}M "
          f"(range: ${ranges[idx]:.1f}M)")

---
## 14. Model Integrity Checks

A model without integrity checks is like a bridge without inspections. The two fundamental checks are:

### 14.1 Balance sheet identity

$$\text{Total Assets} = \text{Total Liabilities} + \text{Equity}$$

This must hold in **every projected year**, to the penny (or in our case, to machine precision).

### 14.2 Cash flow reconciliation

$$\text{Cash}_{t} = \text{Cash}_{t-1} + \text{CFO}_t + \text{CFI}_t + \text{CFF}_t$$

The change in cash on the balance sheet must exactly match the net cash flow from the cash flow statement.

### 14.3 Additional checks

- **Retained earnings roll-forward:** $\text{Equity}_t = \text{Equity}_{t-1} + \text{NI}_t - \text{Dividends}_t$
- **PP&E roll-forward:** $\text{PP\&E}_t = \text{PP\&E}_{t-1} + \text{Capex}_t - \text{D\&A}_t$
- **Sign checks:** Cash should be non-negative (unless a revolver is the plug), margins should be within reasonable bounds

> **Key Concept:** Professional financial modellers run integrity checks *automatically* after every model change. If a check fails, the model is broken, and all outputs are unreliable. These checks are the financial modelling equivalent of unit tests.

> **CFA Exam Tip:** If an exam question asks you to identify an error in a financial model, check the balance sheet identity first, then cash flow reconciliation. These two tests catch the vast majority of modelling errors.

In [ ]:
# ── Comprehensive model integrity checks ──
print("=" * 60)
print("       MODEL INTEGRITY CHECKS -- BASE CASE")
print("=" * 60)

all_passed = True

# Check 1: Balance sheet identity
print("\n1. Balance Sheet Identity (A = L + E):")
for i in range(n_proj):
    diff = base['total_assets'][i] - base['total_le'][i]
    passed = abs(diff) < ATOL
    all_passed &= passed
    symbol = 'PASS' if passed else 'FAIL'
    print(f"   Y{years_proj[i]}: A={base['total_assets'][i]:.4f}, "
          f"L+E={base['total_le'][i]:.4f}, Diff={diff:.2e} [{symbol}]")

# Check 2: Cash flow reconciliation
print("\n2. Cash Flow Reconciliation:")
cash_arr = np.concatenate([[cash_hist[-1]], base['cash']])
for i in range(n_proj):
    bs_change = cash_arr[i+1] - cash_arr[i]
    cf_change = base['net_cash_change'][i]
    diff = bs_change - cf_change
    passed = abs(diff) < ATOL
    all_passed &= passed
    symbol = 'PASS' if passed else 'FAIL'
    print(f"   Y{years_proj[i]}: BS change={bs_change:.4f}, "
          f"CF change={cf_change:.4f}, Diff={diff:.2e} [{symbol}]")

# Check 3: Equity roll-forward
print("\n3. Equity Roll-forward:")
eq_arr = np.concatenate([[equity_hist[-1]], base['equity']])
for i in range(n_proj):
    implied = eq_arr[i] + base['net_income'][i] - base['dividends'][i]
    diff = implied - base['equity'][i]
    passed = abs(diff) < ATOL
    all_passed &= passed
    symbol = 'PASS' if passed else 'FAIL'
    print(f"   Y{years_proj[i]}: Prev+NI-Div={implied:.4f}, "
          f"Actual={base['equity'][i]:.4f}, Diff={diff:.2e} [{symbol}]")

# Check 4: PP&E roll-forward
print("\n4. PP&E Roll-forward:")
ppe_arr = np.concatenate([[ppe_hist[-1]], base['ppe']])
for i in range(n_proj):
    implied = ppe_arr[i] + base['capex'][i] - base['da'][i]
    diff = implied - base['ppe'][i]
    passed = abs(diff) < ATOL
    all_passed &= passed
    symbol = 'PASS' if passed else 'FAIL'
    print(f"   Y{years_proj[i]}: Prev+Capex-DA={implied:.4f}, "
          f"Actual={base['ppe'][i]:.4f}, Diff={diff:.2e} [{symbol}]")

# Check 5: Sign checks
print("\n5. Sign / Reasonableness Checks:")
cash_positive = np.all(base['cash'] > 0)
margins_ok = (np.all(base['ebit'] / base['revenue'] > 0) and
              np.all(base['ebit'] / base['revenue'] < 0.5))
print(f"   Cash positive all years: {'PASS' if cash_positive else 'FAIL'}")
print(f"   EBIT margins in (0%, 50%): {'PASS' if margins_ok else 'FAIL'}")
all_passed &= cash_positive and margins_ok

print(f"\n{'='*60}")
result_msg = 'ALL CHECKS PASSED' if all_passed else 'SOME CHECKS FAILED'
print(f"  OVERALL: {result_msg}")
print(f"{'='*60}")

---
## 15. References

1. **CFA Institute** (2024). *CFA Program Curriculum Level I: Financial Statement Analysis*. CFA Institute.
2. **Benninga, S.** (2014). *Financial Modeling*, 4th ed. MIT Press. — The standard reference for Excel-based three-statement modelling.
3. **Koller, T., Goedhart, M., & Wessels, D.** (2020). *Valuation: Measuring and Managing the Value of Companies*, 7th ed. Wiley. — McKinsey's approach to integrated modelling and DCF valuation.
4. **Rosenbaum, J. & Pearl, J.** (2020). *Investment Banking: Valuation, LBOs, M&A, and IPOs*, 3rd ed. Wiley. — Practical guide to building financial models in investment banking.
5. **Damodaran, A.** (2012). *Investment Valuation*, 3rd ed. Wiley. — Comprehensive treatment of forecasting and valuation methods.
6. **Palepu, K. & Healy, P.** (2013). *Business Analysis and Valuation Using Financial Statements*, 5th ed. Cengage. — Framework for linking financial analysis to valuation.

---

*This notebook is the capstone of the Financial Statement Analysis series. It integrates concepts from all prior notebooks — income statement analysis, balance sheet mechanics, cash flow derivation, ratio analysis, and earnings quality — into a single, coherent modelling framework. The iterative circular reference resolution and Monte Carlo simulation demonstrate how a deterministic accounting model can be extended into a probabilistic decision tool.*